In [7]:
//N.B. This example serves to show that running QCModAffine
//directly does not work since there are non-trivial
//height contributions

SetLogFile("187star.out");
load "StarQuotientMeasures.m";
load "rank.m";
SetPath("QCMod");
load "qc_modular.m";
load "divisor_heights.m";
load "qc_init_g2.m";
load "howe_zhu.m";
SetPath("QCMod");
load "../ModularCurvesX0plusG4-6/model_equation_finder.m";
load "../misc.m";

//SetDebugOnError(true);

N := 187;
ALs := [11, 17];
seq_al := ALs;

X187star := X0NQuotient(N, ALs);
pts := PointSearch(X187star, 30);
// Ambient projective space
P<X, Y, Z> := ProjectiveSpace(Rationals(), 2);

/*S, Q, model_map, pts187, good_primes, inf_pts, bad_pts :=
find_and_test_model(-Z,Y+Z, Y+Z, X+Y+Z, X187star, x, y, pts);
good_primes;
//[31]
Q;
primes := [31]; // TODO

SetMemoryLimit(10 * 1024^3);*/
 L1:=X + Y + 2*Z;
 L2:=2*X - 2*Y + Z;
 L3:=Y - Z;
 L4:=-2*X + 2*Y - Z;

S, Q, model_map, pts187, good_primes, inf_pts, bad_pts :=
                        find_and_test_model(L1, L2, L3, L4, X187star, x, y, pts
: printlevel:=1);

primes := [37];


J := JZero(N);
if #seq_al gt 0 then
    J_quo := Image(&*[1+AtkinLehnerOperator(J,i) : i in ALs]);
else
    J_quo := J;
end if;
J := J_quo;
gX := Dimension(J);
printf "genus = %o\n", gX;
assert rank_quo(N, ALs) eq gX; // check QC condition

ptsX := PointSearch(S, 30); // projective coordinates
// First compute local heights of representatives for generators of J(Q) tensor Q at p. 
index_of_base_point := [i : i in [1..#ptsX] | ptsX[i,3] ne 0][1]; // first index of ptsX such that the z-coordinate is non-zero
assert ptsX[index_of_base_point,3] ne 0;
//base_pt := [ptsX[index_of_base_point,1]/ptsX[index_of_base_point,3], ptsX[index_of_base_point,2]/ptsX[index_of_base_point,3]^(gX+1)]; 
base_pt := [ptsX[index_of_base_point,1]/ptsX[index_of_base_point,3], ptsX[index_of_base_point,2]/ptsX[index_of_base_point,3]]; 
printf "Using base point %o.\n", base_pt;

// check that differences of ptsX generate a subgroup of rank gX of J(Q)
//torsion_bas, torsion_orders, bas, mMW := generators(J);
//assert #bas eq gX; // rank = 2
torsionMultiple := TorsionMultiple(J);
printf "#J(Q)_tors divides %o.\n", torsionMultiple;
assert torsionMultiple eq 1;

X := S;

// if ([J(Q) : <differences of points>], M) = 1, we can use the differrences as divisors, spliting indices [1,0,...0] and so on and torsion_bas = []

/*// find a finite index subgroup (generated by divisors) of J(Q) given by differences of P in C(Q), and writes them in the basis bas of J(Q) (splitting_indices)
function divisors_from_curve(X)
  divisors_found := [];
  splitting_indices_found := [];
  indices_found := [];
  index := 0;
  for P1P2 in Subsets({P : P in ptsX}, 2) do
    P1, P2 := Explode(Setseq(P1P2));
    if 0 in {P1[3], P2[3]} then // exclude the case where one point is at infinity
      continue;
    end if;
    for Q1Q2 in Subsets({P : P in ptsX}, 2) do
      Q1, Q2 := Explode(Setseq(Q1Q2));
      if 0 in {Q1[3], Q2[3]} or P1P2 eq Q1Q2 then // exclude the case where one point is at infinity
        continue;
      end if;
      divisors := [* [* [[P1[1]/P1[3],P1[2]/P1[3]^(gX+1)]], [[P2[1]/P2[3],P2[2]/P2[3]^(gX+1)]] *],
                     [* [[Q1[1]/Q1[3],Q1[2]/Q1[3]^(gX+1)]], [[Q2[1]/Q2[3],Q2[2]/Q2[3]^(gX+1)]] *]*];
      D1 := P1 - P2; // in J(Q)
      D2 := Q1 - Q2;
      splitting_generators := [D1, D2];
      
      D1_in_MW := D1 @@ mMW;
      D2_in_MW := D2 @@ mMW;
      splitting_indices := [Eltseq(D1_in_MW)[#torsion_bas+1..#torsion_bas+gX], Eltseq(D2_in_MW)[#torsion_bas+1..#torsion_bas + #bas]]; // D1, D2 in bas.i
      
      // compute the index of the span of divisors in the MW group
      index := Index(Domain(mMW), sub< Domain(mMW) | [D1_in_MW, D2_in_MW] >);
      if index ne 0 then
        Append(~divisors_found, divisors);
        Append(~splitting_indices_found, splitting_indices);
        Append(~indices_found, index);
      end if;
      if index eq &*torsion_orders then // we found the whole free part of the MW group
        break;
      end if;
    end for;
    if index eq &*torsion_orders then
      break;
    end if;
  end for;
  // take the configuration with minimal finite index
  min, pos := Minimum(indices_found);
  assert min gt 0;
  divisors := divisors_found[pos];
  splitting_indices := splitting_indices_found[pos];
  printf "Use independent points %o\n", divisors;
  printf "splitting indices: %o\n", splitting_indices;
  printf "index in J(Q): %o\n", min;
  return divisors, splitting_indices;
end function;

divisors, splitting_indices := divisors_from_curve(X);*/
// TODO: check G/M = J(Q)/M
pts := [ptsX[i] : i in [1..#ptsX] | i ne index_of_base_point];
base_point := ptsX[index_of_base_point];
//divisors := [* [* [[P[1]/P[3],P[2]/P[3]^(gX+1)]], [[base_point[1]/base_point[3],base_point[2]/base_point[3]^(gX+1)]] *]
//                    : P in pts *];
divisors := [* [* [[P[1]/P[3],P[2]/P[3]]], [[base_point[1]/base_point[3],base_point[2]/base_point[3]]] *]
                    : P in pts *];
assert gX eq 3;
assert torsionMultiple eq 1;
splitting_indices := [ [1,0,0], [0,1,0], [0,0,1] ];

function run_QC(p, N1, N2)
  assert N1 ge N2;

  try
    //splitting_generators, divisors, intersections, splitting_indices, odd_divisors_Qp := height_init_g2(X, p, bas: N := N1, multiple_bound := 40); 
    printf "Computing Coleman data with N1 = %o.\n", N1;
    data := coleman_data(Q, p, N1 : useU := false);

    printf "\nStarting quadratic Chabauty for p = %o.\n", p;
    R<x> := PolynomialRing(Rationals());
    //use_polys := [x + 1, x^2 - 27]; // p = 31
    corrs := StarQuotientGoodCorrespondences(187, [p]);
    polys := corrs[1];
    time good_affine_rat_pts_xy, no_fake_pts, bad_affine_rat_pts_xy, data, fake_rat_pts, bad_Qppoints := QCModAffine(Q, p : use_polys := polys,
        number_of_correspondences := #polys-1, printlevel:=1, N := N1, prec := 40, base_point := base_pt);
    printf "There are %o fake rational points: %o\n", #fake_rat_pts, fake_rat_pts;
    printf "There are %o bad Q_%o-points.\n", #bad_Qppoints, p;

    function recompute_Coleman_data(N1, N2, fake_rat_pts)
      try
        // BUG: recompute Coleman data
        // Here * good_affine_rat_pts_xy contains the found rational points in disks where the Frob lift is defined 
        //      * no_fake_pts is true iff the solutions are exactly the rational points
        //      * bad_affine_rat_pts_xy contains the found rational points in disks where the Frob lift isn't defined 
        //      * data is the Coleman data of X at p used in the qc computation
        //      * fake_rat_pts contains the p-adic solutions that don't look rational
        //      * bad_Qppoints contains the disks where Frob isn't defined
        //
        // Express the images of the solutions under Abel-Jacobi in terms of the generators mod p^N
        fake_rat_pts_new := [];
        for i in [1..#fake_rat_pts] do
          fake_rat_pts_new[i] := [ChangePrecision(fake_rat_pts[i,j], N2) : j in [1..2]]; // N2 div 2
          // lower precision for speed and to avoid issues in Coleman integrals.
        end for;
        printf "Re-computing Coleman data with N2 = %o.\n", N2;
        data := coleman_data(Q, p, N2 : useU := false);

        fake_coeffs_mod_pN, rat_coeffs_mod_pN := coefficients_mod_pN(fake_rat_pts_new, good_affine_rat_pts_xy, divisors, base_pt, splitting_indices, data : printlevel := 3); 
        // Check that the coefficients of the known rational points are correct.
        //assert &and[&+[rat_coeffs_mod_pN[j,i] * bas[i] : i in [1..gX]] eq X!good_affine_rat_pts_xy[j] - X!base_pt : j in [1..#good_affine_rat_pts_xy]];
      catch e 
        printf "%o\n", e;
        if N2 lt N1 then
          N2 +:= 1;
        else 
          error "reached N2 = N1";
        end if;
        printf "trying N1 = %o, N2 = %o ...\n", N1, N2;
        return $$(N1, N2, fake_rat_pts);
      end try;
      return fake_coeffs_mod_pN, rat_coeffs_mod_pN;
    end function;

    fake_coeffs_mod_pN, rat_coeffs_mod_pN := recompute_Coleman_data(N1, N2, fake_rat_pts);
  catch e
    N1 +:= 1;
    return $$(primes, N1, 3);
  end try;
  
  return fake_coeffs_mod_pN, rat_coeffs_mod_pN;
end function;

fake_coeffs := [];
rat_coeffs := [];
for p in primes do
  fake_coeffs_mod_pN, rat_coeffs_mod_pN := run_QC(p, 20, 8);
  Append(~fake_coeffs, [fake_coeffs_mod_pN]);
  Append(~rat_coeffs, [rat_coeffs_mod_pN]);
end for;

print "done.";

Loading "StarQuotientMeasures.m"
Loading "rank.m"
Loading "QCMod/qc_modular.m"
Loading "QCMod/coleman.m"
Loading "QCMod/auxpolys.m"
Loading "QCMod/coho.m"
Loading "QCMod/froblift.m"
Loading "QCMod/reductions.m"
Loading "QCMod/singleintegrals.m"
Loading "misc.m"
Loading "QCMod/applications.m"
Loading "QCMod/symplectic_basis.m"
Loading "QCMod/hecke_correspondence.m"
Loading "QCMod/hodge.m"
Loading "QCMod/frobenius.m"
Loading "QCMod/heights.m"
Loading "QCMod/second_patch_quartic.m"
Loading "QCMod/divisor_heights.m"
Loading "QCMod/qc_init_g2.m"
Loading "mws_qc.m"
Loading "QCMod/howe_zhu.m"
Loading "QCMod/../ModularCurvesX0plusG4-6/model_equation_finder.m"
Loading "QCMod/qc_modular.m"
Loading "QCMod/coleman.m"
Loading "QCMod/auxpolys.m"
Loading "QCMod/coho.m"
Loading "QCMod/froblift.m"
Loading "QCMod/reductions.m"
Loading "QCMod/singleintegrals.m"
Loading "misc.m"
Loading "QCMod/applications.m"
Loading "QCMod/symplectic_basis.m"
Loading "QCMod/hecke_correspondence.m"
Loading "QCMod/hodge.m"

In [ ]:
load "mws_qc.m";
pts := [ptsX[i] : i in [1..#ptsX] | i ne index_of_base_point];
base_point := ptsX[index_of_base_point];

printf "\nUse the Mordell-Weil sieve to show that the additional %o solutions aren't rational.\n", [#fake_coeffs_mod_pN[1] : fake_coeffs_mod_pN in fake_coeffs];
printf "generating cosets ...\n";
exponents := [Integers()| 1 : i in [1..#primes]]; // TODO
qc_fake_coeffs_mod_M := coeffs_CRT(fake_coeffs, primes, exponents);
printf "number of cosets: %o\n", #qc_fake_coeffs_mod_M;
qc_M := &*[primes[i]^exponents[i] : i in [1..#primes]];  // modulus
M := qc_M;
aux_int := 1; //3^3 * 5; // TODO
printf "adding information modulo %o\n", aux_int;
fake_coeffs_mod_M := combine_fake_good(qc_fake_coeffs_mod_M, qc_M, aux_int);
M := qc_M*aux_int; // modulus
qc_rat_coeffs_mod_M := [];
for i in [1..#ptsX] do
  //ptJ := X!pt - X!base_pt;
  //Append(~qc_rat_coeffs_mod_M, [t mod M : t in Eltseq(ptJ@@mMW)]);
  Append(~qc_rat_coeffs_mod_M, [0 : j in [1..i-1]] cat [1] cat [0 : j in [i+1..gX]]);
end for;
printf "number of cosets: %o\n", #fake_coeffs_mod_M;
mws_primes := {73}; //sieving_primes(M, good_primes, groups, 70); // TODO
printf "sieving with S = %o\n", mws_primes;
//printf "gcd(#J(F_ell), #J(F_q)) (ell in S, q | MM') = %o\n", Gcd([#BaseChange(J, GF(v)) : v in mws_primes cat PrimeDivisors(M)]);
//factored_orders := [FactoredOrder(BaseChange(J, GF(v))) : v in mws_primes];
//printf "with [#J(F_v) : v in S] = %o\nand gcd_v(#J(F_v)) = %o\n", factored_orders, Factorization(Gcd([#BaseChange(J, GF(v)) : v in mws_primes]));

bas := [[Eltseq(pt), Eltseq(base_point)] : pt in pts][1..gX]; // TODO: assumes we have a free generating set

d := 1;

printf "starting Mordell Weil sieve\n";
X := Curve(P, DefiningEquations(S)[1]);
time done_fake := MWSieve(X, mws_primes, M, bas, base_point, fake_coeffs_mod_M : 
  known_rat_coeffs := qc_rat_coeffs_mod_M, d := d, excluded := PrimeDivisors(N), printlevel := 2);

printf "done with the MW sieve: %o\n", done_fake;
assert done_fake;
printf "No additional solutions are rational.\n";

Loading "mws_qc.m"

Use the Mordell-Weil sieve to show that the additional [ 54 ] solutions aren't rational.
generating cosets ...
number of cosets: 54
adding information modulo 1
number of cosets: 54
sieving with S = { 73 }
starting Mordell Weil sieve
 GroupInfo: p = 73...
   #C(F_p) = 86, Invariants(G) = [ 474238 ]
   Exponent = [ <2, 1>, <31, 1>, <7649, 1> ]
   Group Structure = Abelian Group isomorphic to Z/474238
Defined on 1 generator in supergroup:
G.1 = $.1
Relations:
474238*G.1 = 0
   Look at 1-multiples 
make DL
starting DL of generators
finished DL of generators
starting DL of points on curve over F_p
finished DL
        54 cosets remaining after v=73
Time: 0.370
done with the MW sieve: false

>> assert done_fake;
   ^
Runtime error in assert: Assertion failed
No additional solutions are rational.


In [28]:
P<X, Y, Z> := ProjectiveSpace(Rationals(), 2);
X := Curve(P, DefiningEquations(S)[1]);
X := BaseChange(X, Bang(Rationals(),GF(5)));
G, m, minv := ClassGroup(X);
pts := Places(X,1);
minv(pts[1] - pts[2]);
print m;

550*G.1
Mapping from: GrpAb: G to Group of divisors of X


In [30]:
b := bas[1];
S!b[1];
X!ChangeUniverse(Eltseq(b[1]), GF(5));

(1/2 : 0 : 1)
(3 : 0 : 1)


In [16]:
TorsionSubgroup($1);

Abelian Group isomorphic to Z/474238
Defined on 1 generator in supergroup:
$.1 = $.1
Relations:
474238*$.1 = 0
